In [ ]:
!pip install -q transformers bitsandbytes accelerate evaluate bert_score sentence-transformers

In [ ]:
import pandas as pd
import json
import torch
import re
import traceback

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sklearn.model_selection import GroupShuffleSplit
from evaluate import load
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

In [ ]:
def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        trust_remote_code=True
    )

    return tokenizer, model

In [ ]:
model_name = "google/gemma-3-1b-it"

tokenizer, model = load_model(model_name)

Prepare the dataset

In [ ]:
def clean_punctuation(text):
    if not isinstance(text, str):
        return text

    text = re.sub(r"\s'(\w)", r"'\1", text)
    text = re.sub(r"\s([.,!?;:])", r"\1", text)
    text = text.replace(" n't", "n't")
    return text.strip()

In [ ]:
df = pd.read_csv("ru_idioms_corpus.csv")

text_columns = ['Literal_Sent', 'Idiomatic_Sent', 'Idiom']
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].apply(clean_punctuation)

gss_test = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_val_idx, test_idx = next(gss_test.split(df, groups=df['Idiom']))

train_val_df = df.iloc[train_val_idx]
test_df = df.iloc[test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.11, random_state=42)
train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df['Idiom']))

train_df = train_val_df.iloc[train_idx]
val_df = train_val_df.iloc[val_idx]

print(f"Idioms in Train: {train_df['Idiom'].nunique()}, Lines: {len(train_df)}")
print(f"Idioms in Val: {val_df['Idiom'].nunique()}, Lines: {len(val_df)}")
print(f"Idioms in Test: {test_df['Idiom'].nunique()}, Lines: {len(test_df)}")

Hard Prompting

In [ ]:
class IdiomPromptGeneratorEnglish:
    def __init__(self, train_df):
        self.train_df = train_df
        self.system_prompt_default = (
            "You are a linguistic expert specializing in English idioms. "
            "Your task is to transform sentences according to instructions. "
            "Output ONLY the requested sentence. Never provide explanations."
        )

    def get_few_shot_examples(self, direction='to_idiomatic', n=3, exclude_idiom=None):
        if exclude_idiom:
            available_samples = self.train_df[self.train_df['Idiom'] != exclude_idiom]
        else:
            available_samples = self.train_df

        samples = available_samples.sample(n)
        examples_str = ""
        for _, row in samples.iterrows():

            if direction == 'to_idiomatic':
                examples_str += f"Sentence: {row['Literal_Sent']}.\nOutput: {row['Idiomatic_Sent']}\n\n"
            elif direction == 'to_literal':
                examples_str += f"Sentence: {row['Idiomatic_Sent']}.\nOutput: {row['Literal_Sent']}\n\n"
        return examples_str

    def create_prompt(self, target_row, strategy='zero_shot', direction='to_idiomatic'):
        literal = target_row['Literal_Sent']
        idiomatic = target_row['Idiomatic_Sent']
        current_idiom = target_row['Idiom']

        sys_p = self.system_prompt_default

        if direction == 'to_idiomatic':
            if strategy == 'zero_shot':
                prompt = (f"Rewrite the sentence by incorporating an idiom.\n"
                          f"Sentence for you: {literal}.\n"
                          f"Your output:")

            elif strategy == 'few_shot':
                examples = self.get_few_shot_examples(direction, n=3, exclude_idiom=current_idiom)
                prompt = (f"Transform the sentence by incorporating an idiom as shown below:\n\n{examples}"
                          f"\n\nSentence for you: {literal}\n"
                          f"Your output:")

        elif direction == 'to_literal':
            if strategy == 'zero_shot':
                prompt = (f"Replace an idiom in the sentence with a literal expression.\n"
                          f"Sentence for you: {idiomatic}\n"
                          f"Your output:")

            elif strategy == 'few_shot':
                examples = self.get_few_shot_examples(direction, n=3, exclude_idiom=current_idiom)
                prompt = (f"Simplify the sentence by replacing an idiom as shown below:\n\n{examples}"
                          f"Sentence for you: {idiomatic}\n"
                          f"Your output:")

        return sys_p, prompt

In [ ]:
class IdiomPromptGeneratorRussian:
    def __init__(self, train_df):
        self.train_df = train_df
        self.system_prompt_default = (
            "Ты — эксперт по лингвистике, специализирующийся на русских идиомах. "
            "Ты должен перефразировать предложения согласно инструкциям. "
            "Выводи ТОЛЬКО запрошенное предложение. Не пиши никаких пояснений."
        )

    def get_few_shot_examples(self, direction='to_idiomatic', n=3, exclude_idiom=None):
        if exclude_idiom:
            available_samples = self.train_df[self.train_df['Idiom'] != exclude_idiom]
        else:
            available_samples = self.train_df

        samples = available_samples.sample(n)
        examples_str = ""
        for _, row in samples.iterrows():

            if direction == 'to_idiomatic':
                examples_str += f"Предложение: {row['Literal_Sent']}\nВывод: {row['Idiomatic_Sent']}\n\n"
            elif direction == 'to_literal':
                examples_str += f"Предложение: {row['Idiomatic_Sent']}\nВывод: {row['Literal_Sent']}\n\n"
        return examples_str

    def create_prompt(self, target_row, strategy='zero_shot', direction='to_idiomatic'):
        literal = target_row['Literal_Sent']
        idiomatic = target_row['Idiomatic_Sent']
        current_idiom = target_row['Idiom']

        sys_p = self.system_prompt_default

        if direction == 'to_idiomatic':
            if strategy == 'zero_shot':
                prompt = (f"Перепиши предложение, добавив идиому.\n"
                          f"Предложение для тебя: {literal}\n"
                          f"Твой вывод:")

            elif strategy == 'few_shot':
                examples = self.get_few_shot_examples(direction, n=3, exclude_idiom=current_idiom)
                prompt = (f"Перефразируй предложение, добавив идиому, как показано ниже:\n\n{examples}"
                          f"Предложение для тебя: {literal}\n"
                          f"Твой вывод:")

        elif direction == 'to_literal':
            if strategy == 'zero_shot':
                prompt = (f"Замени идиому в предложении буквальным выражением.\n"
                          f"Предложение для тебя: {idiomatic}\n"
                          f"Твой вывод:")

            elif strategy == 'few_shot':
                examples = self.get_few_shot_examples(direction, n=3, exclude_idiom=current_idiom)
                prompt = (f"Упрости предложение, заменив идиому, как показано ниже:\n\n{examples}"
                          f"Предложение для тебя: {idiomatic}\n"
                          f"Твой вывод:")

        return sys_p, prompt

In [ ]:
def run_experiment(model, tokenizer, test_df, generator, strategy='zero_shot',
                   direction='to_idiomatic'):

    raw_model_name = getattr(model.config, "_name_or_path", "unknown_model")
    model_name = raw_model_name.split('/')[-1]

    base_dir = "hard_prompting_results"
    model_dir = os.path.join(base_dir, model_name)
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)

    filename = f"results_{strategy}_{direction}.csv"
    save_path = os.path.join(model_dir, filename)

    results_list = []

    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Running {filename}"):
        sys_p, user_p = generator.create_prompt(row, strategy=strategy,
                                                direction=direction)

        messages = [
            {"role": "system", "content": sys_p},
            {"role": "user", "content": user_p},
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        input_length = inputs["input_ids"].shape[-1]
        new_tokens = output_ids[0][input_length:]
        generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        results_list.append({
            'original_index': idx,
            'idiom': row['Idiom'],
            'input_text': row['Literal_Sent'] if direction == 'to_idiomatic' else row['Idiomatic_Sent'],
            'reference': row['Idiomatic_Sent'] if direction == 'to_idiomatic' else row['Literal_Sent'],
            'generated_text': generated_text,
            'strategy': strategy,
        })

    final_df = pd.DataFrame(results_list)
    final_df.to_csv(save_path, index=False)
    return save_path, model_dir

In [ ]:
def calculate_metrics(csv_path, ref_lookup, lang="en"):
    df = pd.read_csv(csv_path)

    meteor_metric = load("meteor")
    bertscore_metric = load("bertscore")
    bert_score_model = "roberta-large" if lang == "en" else "xlm-roberta-large"

    if lang == "en":
        sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    else:
        sbert_model = SentenceTransformer('sentence-transformers/LaBSE')


    preds = df['generated_text'].fillna("").tolist()
    inputs = df['input_text'].tolist()

    refs = [ref_lookup.get(inp, [df.iloc[i]['reference']]) for i, inp in enumerate(inputs)]

    meteor_res = meteor_metric.compute(predictions=preds, references=refs)

    all_bs_f1 = []
    for p, rs in zip(preds, refs):
        res = bertscore_metric.compute(predictions=[p] * len(rs), references=rs, lang=lang, model_type=bert_score_model)
        all_bs_f1.append(max(res['f1']))

    max_cosines = []
    for pred, rs in zip(preds, refs):
        p_emb = sbert_model.encode([pred], convert_to_tensor=True)
        r_embs = sbert_model.encode(rs, convert_to_tensor=True)
        scores = util.cos_sim(p_emb, r_embs)
        max_cosines.append(scores.max().item())

    return {
        "METEOR": round(meteor_res['meteor'], 4),
        "BERTScore_F1": round(sum(all_bs_f1) / len(all_bs_f1), 4),
        "Cosine_Similarity": round(sum(max_cosines) / len(max_cosines), 4),
    }

In [ ]:
def evaluate_and_append_stats(csv_path, ref_lookup, strategy, direction, model_dir, lang="en"):
    results = calculate_metrics(csv_path, ref_lookup, lang=lang)

    results.update({
        "strategy": strategy,
        "direction": direction,
        "filename": os.path.basename(csv_path)
    })

    stats_path = os.path.join(model_dir, "statistics.csv")

    stats_df = pd.DataFrame([results])
    if not os.path.exists(stats_path):
        stats_df.to_csv(stats_path, index=False)
    else:
        existing_stats = pd.read_csv(stats_path)
        updated_stats = pd.concat([existing_stats, stats_df]).drop_duplicates(
            subset=['strategy', 'direction'], keep='last'
        )
        updated_stats.to_csv(stats_path, index=False)

    return results

In [ ]:
generator = IdiomPromptGeneratorRussian(train_df)

In [ ]:
ref_to_idiom = train_df.groupby('Literal_Sent')['Idiomatic_Sent'].apply(list).to_dict()
ref_to_literal = train_df.groupby('Idiomatic_Sent')['Literal_Sent'].apply(list).to_dict()

In [ ]:
strategies = ['zero_shot', 'few_shot']
directions = ['to_idiomatic', 'to_literal']

for strategy in strategies:
    for direction in directions:
          print(f"\nProcessing: {strategy} | {direction}")

          try:
              csv_path, model_dir = run_experiment(
                  model=model,
                  tokenizer=tokenizer,
                  test_df=train_df,
                  generator=generator,
                  strategy=strategy,
                  direction=direction
              )

              lookup = ref_to_idiom if direction == 'to_idiomatic' else ref_to_literal

              print(f"Calculating metrics for {os.path.basename(csv_path)}")
              evaluate_and_append_stats(csv_path, lookup, strategy, direction, model_dir, lang="ru")

            except Exception as e:
                print(f"Error in {strategy}_{direction}:")
                traceback.print_exc()